# 09 · RAG — Naive to Advanced

The model doesn't know your data. **RAG** fixes that: retrieve the relevant pieces, put them in the prompt.

```
INGEST:  docs ─► split ─► embed ─► vector store
QUERY:   question ─► similarity search ─► top-k chunks ─► prompt ─► LLM ─► answer
```

We index docs for **Aurora, a fictional workflow engine** — fictional so the model *can't* answer from training data. Then: naive RAG → where it fails → three upgrades (query rewriting, score thresholds, citations).

---

## Setup

In [ ]:
%pip install -qU langchain langchain-openai langchain-text-splitters python-dotenv

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv("../../.env")

llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0)
embeddings = init_embeddings(os.environ["EMBEDDING_MODEL"], provider=os.environ["MODEL_PROVIDER"])

---
## 1. The corpus

A few pages of documentation, each with a `source` in its metadata (we'll need that for citations).

In [ ]:
docs = [
    Document(
        page_content=(
            "Aurora is an open-source workflow orchestration engine written in Rust. "
            "Workflows are called 'flows' and are defined either in YAML or with the Python SDK. "
            "Aurora 3.0 was released in March 2026 and requires Python 3.11 or newer for the SDK."
        ),
        metadata={"source": "overview.md"},
    ),
    Document(
        page_content=(
            "Install the Aurora SDK with `pip install aurora-engine`. Start a local dev server "
            "with `aurora dev`; it runs flows on your machine with hot reload. For production, "
            "deploy the Aurora server via the official Docker image or the Helm chart."
        ),
        metadata={"source": "install.md"},
    ),
    Document(
        page_content=(
            "Aurora scales horizontally: a scheduler distributes flow runs across worker nodes, "
            "and a cluster supports up to 512 workers. If a worker crashes mid-run, checkpoint "
            "recovery resumes the flow from the last completed step on another worker — no rerun "
            "of finished steps, no manual intervention."
        ),
        metadata={"source": "scaling.md"},
    ),
    Document(
        page_content=(
            "Aurora itself is free and MIT-licensed. Aurora Cloud, the managed offering, costs "
            "$99 per seat per month and includes hosted workers, the observability dashboard, "
            "and SSO. A free tier covers 1,000 flow runs per month."
        ),
        metadata={"source": "pricing.md"},
    ),
    Document(
        page_content=(
            "Secrets in Aurora are injected at runtime and never stored in flow definitions. "
            "Native integrations exist for HashiCorp Vault and AWS Secrets Manager; rotate keys "
            "without redeploying flows. All secret access is written to the audit log."
        ),
        metadata={"source": "security.md"},
    ),
    Document(
        page_content=(
            "Every flow run emits structured logs and OpenTelemetry traces. The `aurora logs` "
            "command tails a run in real time. Failed runs keep their full state for 30 days, "
            "so you can inspect inputs and outputs of every step after the fact."
        ),
        metadata={"source": "observability.md"},
    ),
]

---
## 2. Split into chunks

Embeddings capture *one idea* well — whole documents blur. Overlap protects ideas that straddle a cut.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print(f"{len(docs)} docs → {len(chunks)} chunks")
print(f"e.g. [{chunks[0].metadata['source']}] {chunks[0].page_content[:90]}...")

---
## 3. Embed + index, then search

`InMemoryVectorStore` (langchain-core, no extra deps) is perfect for learning — production stores (Chroma, pgvector, Pinecone...) share the exact same interface.

In [ ]:
vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)

for d in vectorstore.similarity_search("How much does the managed version cost?", k=2):
    print(f"[{d.metadata['source']}] {d.page_content[:90]}...")

---
## 4. Naive RAG: the whole pipeline as one LCEL chain

In [ ]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template(
    "Answer using ONLY the context below.\n\nContext:\n{context}\n\nQuestion: {question}"
)

rag = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag.invoke("How do I install Aurora and run it locally?"))

Works — for questions phrased like the docs. Now the failure modes.

---
## 5. Upgrade 1 — Query rewriting (multi-query)

Users don't speak documentation. *"My flows die when a machine falls over"* ≠ *"checkpoint recovery resumes interrupted runs"*. Let the LLM generate rephrasings, retrieve for each, merge.

In [ ]:
class Rewrites(BaseModel):
    """Alternative phrasings of a search query."""
    queries: list[str] = Field(description="3 diverse rephrasings of the question")

rewriter = (
    ChatPromptTemplate.from_template(
        "Generate 3 diverse rephrasings of this question for searching technical docs. "
        "Vary the vocabulary.\n\nQuestion: {question}"
    )
    | llm.with_structured_output(Rewrites)
)

def multi_query_retrieve(question: str, k: int = 2):
    queries = [question] + rewriter.invoke({"question": question}).queries
    seen, results = set(), []
    for q in queries:
        for d in vectorstore.similarity_search(q, k=k):
            if d.page_content not in seen:
                seen.add(d.page_content)
                results.append(d)
    print("searched with:", *[f"  - {q}" for q in queries], sep="\n")
    return results

question = "What happens to my flows when a machine falls over?"
found = multi_query_retrieve(question)
print("\nretrieved:", [d.metadata["source"] for d in found])

In [ ]:
answer = llm.invoke(
    prompt.format_messages(context=format_docs(found), question=question)
)
print(answer.content)

---
## 6. Upgrade 2 — Score threshold: "I don't know" beats a lie

Similarity search **always** returns k chunks, relevant or not. Look at the scores (here: cosine similarity, higher = closer):

In [ ]:
for q in ["How many workers can a cluster have?",      # on-topic
          "What is the capital of France?"]:            # off-topic
    best_doc, best_score = vectorstore.similarity_search_with_score(q, k=1)[0]
    print(f"{best_score:.3f}  [{best_doc.metadata['source']}]  ← {q}")

In [ ]:
THRESHOLD = 0.35   # calibrate on your own on-topic vs. off-topic questions

def guarded_rag(question: str) -> str:
    hits = vectorstore.similarity_search_with_score(question, k=3)
    if hits[0][1] < THRESHOLD:
        return "I can't answer that from the Aurora documentation."
    context = format_docs([d for d, _ in hits])
    return (prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )

print(guarded_rag("How many workers can a cluster have?"))
print(guarded_rag("What is the capital of France?"))

> One `if` statement removed a whole class of hallucinations.

---
## 7. Upgrade 3 — Citations

The chunks already carry `metadata` — put the source into the context and ask for it back.

In [ ]:
cite_prompt = ChatPromptTemplate.from_template(
    "Answer using ONLY the context. Cite the source file for each claim, like [pricing.md].\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def cited_rag(question: str) -> str:
    docs = retriever.invoke(question)
    context = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)
    return (cite_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )

print(cited_rag("Is Aurora free? What does the paid version add?"))

---
## What to remember

| Concept | What it does |
|---|---|
| split → embed → store → retrieve → generate | The RAG pipeline; ingest once, query per question |
| `RecursiveCharacterTextSplitter(300, 50)` | Chunk on natural boundaries; overlap saves cut ideas |
| `vectorstore.as_retriever()` | Vector store → Runnable → pipes straight into LCEL |
| Multi-query rewrite | Recall stops depending on the user's word choice |
| `similarity_search_with_score` + threshold | Refuse instead of hallucinate |
| Sources in context | Verifiable answers, debuggable retrieval |

**Further up the ladder:** hybrid search (BM25 + vectors), rerankers, parent-document retrieval, and *agentic RAG* — lesson 06's agent with retrieval as a tool.

**Next:** 10 — **Multi-agent and subgraph patterns**: when one agent isn't enough.